# 02 — Pose Landmark Extraction

Purpose: Run MediaPipe Pose Landmarker on the front photo, extract 33 key points, visualize them and check per-landmark confidence (visibility).

Prerequisites:
- models/pose_landmarker_heavy.task is downloaded (see README section 4)
- Images exist under data/raw/front/

GPU note:
- The MediaPipe Python wheel on Windows has limited GPU delegate support, so we run with delegate="cpu".
- Inference is well under 100 ms per image at PoC scale.

In [ ]:
%load_ext autoreload
%autoreload 2

from pathlib import Path
import sys, json
import cv2
import numpy as np

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

from src.pose_detection import build_pose_landmarker, detect_pose, LANDMARK_NAMES
from src.visualization import plot_pose_overlay

MODEL_PATH = ROOT / "models" / "pose_landmarker_heavy.task"
FRONT_DIR = ROOT / "data" / "raw" / "front"
OUT_DIR   = ROOT / "data" / "processed" / "landmarks"
OUT_DIR.mkdir(parents=True, exist_ok=True)
MODEL_PATH, FRONT_DIR

In [ ]:
landmarker = build_pose_landmarker(MODEL_PATH, delegate="cpu")

images = sorted(FRONT_DIR.glob("*.jp*g")) + sorted(FRONT_DIR.glob("*.png"))
print(f"{len(images)} image(s) found.")
images[:5]

In [ ]:
import matplotlib.pyplot as plt

for img_path in images:
    bgr = cv2.imread(str(img_path))
    rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
    pose = detect_pose(landmarker, rgb)
    print(f"{img_path.name}: detected={pose.detected}")
    if pose.detected:
        # save landmarks as JSON for downstream notebooks
        out_json = OUT_DIR / f"{img_path.stem}.json"
        out_json.write_text(json.dumps({
            "image": str(img_path),
            "width": pose.image_width,
            "height": pose.image_height,
            "landmarks_px": pose.landmarks_px.tolist(),
            "landmarks_world": pose.landmarks_world.tolist(),
            "landmark_names": LANDMARK_NAMES,
        }, indent=2))
    fig = plot_pose_overlay(rgb, pose, title=img_path.name)
    plt.show()